# Experiment 11: PolyGuardPrompts + M-ALERT Multilingual Safety (E4)

**Reviewer concern (R1, R2):** the original §14.4 multilingual experiment used 15 self-translated prompts per language, which R1 said is too small to support cross-language ordering claims.

**This notebook:** evaluates the same 4 SLMs (Qwen 2.5-3B, Llama 3.2-3B, Qwen 3-4B, Phi-4-Mini) on:
- 100 prompts/language from PolyGuardPrompts in {Romanian, Chinese, Arabic, Spanish, English}
- 100 prompts/language from M-ALERT in {EN, DE, FR, IT, ES} (overlapping with EN/ES from PolyGuard)

Reports ASR + 95% Wilson CIs per (model, language).

**Output:** `experiments/results/multilingual_polyguard_malert.json`.

**Runtime:** ~2-3 h on A100 (4 models × 5 languages × 200 prompts).


## Setup

In [ ]:
%%capture
!pip install -U 'transformers>=4.51' 'accelerate>=1.1' huggingface_hub datasets ipywidgets statsmodels -q
import os, json, time, gc, hashlib, random
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- Hard-fail if Colab didn't allocate a GPU ---
assert torch.cuda.is_available(), (
    'No GPU detected. In Colab: Runtime > Change runtime type > A100 GPU. '
    'This notebook will not run on CPU.'
)
_gpu_name = torch.cuda.get_device_name(0)
_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'transformers: {transformers.__version__}')
print(f'GPU: {_gpu_name}')
print(f'VRAM: {_vram_gb:.1f} GB')
if 'A100' not in _gpu_name and _vram_gb < 35:
    print()
    print('WARNING: this notebook is configured for an A100 (40 GB).')
    print(f'You appear to have a {_gpu_name} with {_vram_gb:.1f} GB.')
    print('Smaller GPUs may OOM on the 13B HarmBench classifier or the GCG run.')
    print('Consider switching to A100 via Runtime > Change runtime type.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/PhD/paper1-survey')
RESULTS_DIR = DRIVE_ROOT / 'experiments' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Drive mounted at', DRIVE_ROOT)


In [ ]:
# Reproducibility config (do not edit per-run; logged with results)
REPRO = {
    'notebook': None,  # set per-notebook
    'created': datetime.utcnow().isoformat() + 'Z',
    'transformers': transformers.__version__,
    'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'seed': 42,
}
random.seed(REPRO['seed'])
np.random.seed(REPRO['seed'])
torch.manual_seed(REPRO['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(REPRO['seed'])


In [ ]:
REPRO['notebook'] = '11_polyguard_multilingual'
REPRO['polyguard_dataset'] = 'ToxicityPrompts/PolyGuardPrompts'
REPRO['malert_dataset'] = 'Babelscape/M-ALERT'
REPRO['languages_polyguard'] = ['ro', 'zh', 'ar', 'es', 'en']
REPRO['languages_malert'] = ['en', 'de', 'fr', 'it', 'es']
REPRO['n_per_language'] = 100
REPRO['judge'] = 'meta-llama/Llama-Guard-3-1B'
REPRO['models'] = [
    'Qwen/Qwen2.5-3B-Instruct',
    'meta-llama/Llama-3.2-3B-Instruct',
    'Qwen/Qwen3-4B',
    'microsoft/Phi-4-mini-instruct',
]
REPRO


## Load PolyGuardPrompts and M-ALERT samples

Both datasets are public on HuggingFace. We sample `n_per_language` prompts per language with a fixed seed for reproducibility, taking only the prompts labeled as harmful (the benign subset is reserved for over-refusal in another notebook).


In [ ]:
from datasets import load_dataset

def sample_polyguard(lang: str, n: int) -> pd.DataFrame:
    ds = load_dataset(REPRO['polyguard_dataset'], split=lang)
    df = ds.to_pandas()
    df = df[df['harmful'] == True]  # noqa: E712
    return df.sample(n=min(n, len(df)), random_state=REPRO['seed']).reset_index(drop=True)

def sample_malert(lang: str, n: int) -> pd.DataFrame:
    ds = load_dataset(REPRO['malert_dataset'], lang, split='test')
    df = ds.to_pandas()
    return df.sample(n=min(n, len(df)), random_state=REPRO['seed']).reset_index(drop=True)

# Replace with the actual dataset schema once we confirm it on Colab; the code above is the
# template — column names ('prompt', 'harmful', etc.) may need adjustment based on the
# released dataset card. See:
#   - https://huggingface.co/datasets/ToxicityPrompts/PolyGuardPrompts
#   - https://huggingface.co/datasets/Babelscape/M-ALERT
#
# TODO[E4]: verify column names and adjust before first run.


## Generation + judging

We reuse the §14 generation pipeline (greedy decoding for non-Qwen-3 models) and Llama Guard 3-1B as the judge. The prompts are presented in the target language; Llama Guard 3-1B handles multiple languages but its agreement on non-English is part of what we are measuring.


In [ ]:
MAX_NEW_TOKENS = 256

def load_model(model_id: str):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map='auto',
    )
    model.eval()
    return model, tok

def generate(model, tok, prompt: str, do_sample: bool = False) -> str:
    msgs = [{'role': 'user', 'content': prompt}]
    chat = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            chat,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=do_sample,
            temperature=0.7 if do_sample else None,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0, chat.shape[1]:], skip_special_tokens=True)


In [ ]:
# TODO[E4]: full generation + judging loop.
# Pseudocode skeleton:
#
#   results = []
#   guard_model, guard_tok = load_model(REPRO['judge'])
#   for model_id in REPRO['models']:
#       model, tok = load_model(model_id)
#       for lang in REPRO['languages_polyguard']:
#           df = sample_polyguard(lang, REPRO['n_per_language'])
#           for _, row in df.iterrows():
#               resp = generate(model, tok, row['prompt'],
#                               do_sample=('Qwen3' in model_id))
#               label = judge(guard_model, guard_tok, row['prompt'], resp)
#               results.append({'model': model_id, 'dataset': 'polyguard',
#                               'lang': lang, 'prompt': row['prompt'],
#                               'response': resp, 'label': label})
#       del model; gc.collect(); torch.cuda.empty_cache()
#
# Repeat the inner loop for M-ALERT.
# Save per-model partials to results_polyguard_<short>.json for resume support.


## Aggregate ASR with 95% Wilson CIs


In [ ]:
from statsmodels.stats.proportion import proportion_confint

def aggregate(results_df):
    rows = []
    for (model, dataset, lang), g in results_df.groupby(['model', 'dataset', 'lang']):
        n = len(g)
        k_unsafe = (g['label'] == 'unsafe').sum()  # ASR = unsafe rate
        p = k_unsafe / n if n else 0.0
        if n:
            lo, hi = proportion_confint(k_unsafe, n, alpha=0.05, method='wilson')
        else:
            lo = hi = 0.0
        rows.append({'model': model, 'dataset': dataset, 'lang': lang,
                     'n': n, 'asr': p, 'ci_lo': lo, 'ci_hi': hi})
    return pd.DataFrame(rows)

# TODO[E4]: invoke aggregate() once results are populated, save to JSON + CSV.
